# Embeddings

Alle bisherigen Methoden hatten einen gemeinsamen blinden Fleck, den wir in Skript 04 benannt haben: In einer Bag-of-Words-Darstellung sind *Kanzlerin* und *Merkel* oder *Asylwerber* und *Flüchtling* so verschieden, wie zwei Wörter nur sein können. Ähnlichkeit setzt gemeinsame Zeichenketten voraus.

**Embeddings** ersetzen jedes Wort oder jeden Text durch einen dichten Vektor aus einigen hundert Zahlen. Diese Vektoren werden so gelernt, dass Einheiten, die in ähnlichen Kontexten vorkommen, ähnliche Vektoren erhalten. Das ist die *distributionelle Hypothese*: Ein Wort wird durch die Gesellschaft charakterisiert, in der es auftritt. Embedding-Modelle sind neuronale Netze, aber sie sind **Encoder**, keine Generatoren - sie bilden Text auf Vektoren ab und erzeugen selbst keinen Text. Große Sprachmodelle (LLMs) folgen in Skript 07.

Han, Pei und Tong (2023, Kap. 12.1) beschreiben diesen Schritt als Übergang von einer **symbolbasierten** Repräsentation (One-Hot-Vektoren, Bag of Words: jedes Wort eine eigene Dimension, dünn besetzt und sehr hochdimensional) zu einer **verteilten** Repräsentation (jedes Wort ein dichter Vektor mit wenigen hundert Dimensionen, in dem die Bedeutung über alle Dimensionen verteilt ist).

Am Ende dieses Skripts kannst du:

- mit **Wortvektoren** verwandte Wörter finden und erkennen, wo sie danebenliegen;
- **eigene Wortvektoren** mit word2vec (CBOW und Skip-gram) und fastText auf unserem Korpus trainieren;
- **statische** von **kontextuellen** Embeddings unterscheiden, wie sie vortrainierte Sprachmodelle (BERT und Verwandte) liefern;
- **Dokumentvektoren** bilden, zuerst durch Mittelwertbildung, dann mit einem **Sentence Transformer**;
- eine **semantische Suche** durchführen, die Treffer ohne gemeinsame Wörter findet;
- die **Zuordnung von Meldungen** (Story Matching) aus Skript 04 und den **Klassifikator** aus Skript 05 mit Embeddings wiederholen und messen, was sich verändert hat;
- einen Klassifikator **ohne gelabelte Daten** bauen, nur aus den Namen der Klassen (*schwache Überwachung*), und ihn mit **Selbsttraining** verbessern.

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Datentransformation und Visualisierung
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Standardbibliothek
import re
import sys

# Paket für linguistische Annotation und Wortvektoren
import spacy

# Paket zum Trainieren eigener Wortvektoren (word2vec, fastText)
from gensim.models import FastText, Word2Vec

# Pakete für Merkmalsextraktion, Klassifikation und Evaluierung
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.svm import LinearSVC

# Pakete für Sentence Transformer und die Projektion in zwei Dimensionen
import torch
import umap
from sentence_transformers import SentenceTransformer

# Hilfsfunktionen des Kurses
import textmining as tm

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

Die Warnung von `tqdm` (einem Paket für Fortschrittsbalken) ist harmlos: Ohne das Paket `ipywidgets` zeigt Jupyter die Balken als Text statt als Grafik an. Das Ergebnis ist davon nicht betroffen.

## Daten

Wir arbeiten wieder mit den Schlagzeilen von Der Standard und Süddeutscher Zeitung, die wir mit `tm.load_corpus()` laden.

In [ ]:
# Lade das Korpus
corpus = tm.load_corpus()

## Wortvektoren

Die deutschen Modelle von spaCy enthalten statische Wortvektoren: einen 300-dimensionalen Vektor pro Wort, gelernt aus einer großen Sammlung deutscher Texte. "Statisch" bedeutet, dass ein Wort überall, wo es vorkommt, denselben Vektor hat.

Wir beginnen mit einer Falle. Das mittlere Modell aus Skript 02 *hat* Vektoren, aber sieh dir an, wie viele:

In [ ]:
# Lade das mittlere Modell und betrachte seine Vektoren
nlp_md = spacy.load("de_core_news_md")
v = nlp_md.vocab.vectors
print(f"de_core_news_md: {v.n_keys:,} Wörter teilen sich {v.shape[0]:,} verschiedene Vektoren")

# Suche die nächsten Nachbarn von "Kanzler"
keys, _, scores = v.most_similar(np.asarray([nlp_md.vocab["Kanzler"].vector]), n=4)
print("nächste Nachbarn von 'Kanzler':", [(nlp_md.vocab.strings[k], round(float(s), 2)) for k, s in zip(keys[0], scores[0])])

Um klein zu bleiben, bildet das mittlere Modell eine halbe Million Wörter auf zwanzigtausend Vektoren ab. Nicht verwandte Wörter *teilen* sich also einen Vektor und erscheinen als perfekte Nachbarn mit Ähnlichkeit 1.00. Für das, wofür `md` die Vektoren intern verwendet, ist das in Ordnung - für die Untersuchung von Wortbedeutungen ist es nutzlos. Prüfe immer, was deine Vektoren sind, bevor du Schlüsse daraus ziehst.

Das große Modell, `de_core_news_lg`, hat für jedes seiner 500.000 Wörter einen eigenen Vektor. Installiere es einmalig mit `python -m spacy download de_core_news_lg`. Wir benötigen hier nur seine Vektoren, daher lassen wir die übrigen Komponenten der Pipeline weg.

In [ ]:
# Lade das große Modell, nur mit den Komponenten, die wir für die Vektoren brauchen
nlp = spacy.load("de_core_news_lg", exclude=["parser", "ner", "lemmatizer", "attribute_ruler"])
vectors = nlp.vocab.vectors
print(f"de_core_news_lg: {vectors.n_keys:,} Wörter, {vectors.shape[0]:,} verschiedene Vektoren mit {vectors.shape[1]} Dimensionen")


# Finde die n nächsten Nachbarn eines Worts (ohne Varianten in anderer Groß-/Kleinschreibung)
def neighbours(word: str, n: int = 10) -> list[str]:
    keys, _, scores = vectors.most_similar(np.asarray([nlp.vocab[word].vector]), n=n * 3)
    seen, out = {word.casefold()}, []
    for key, score in zip(keys[0], scores[0]):
        w = nlp.vocab.strings[key]
        if w.casefold() not in seen:
            seen.add(w.casefold())
            out.append(f"{w} ({score:.2f})")
    return out[:n]


# Zeige die Nachbarn für einige Beispielwörter
pd.DataFrame({w: neighbours(w) for w in ["Kanzlerin", "Flüchtling", "Börse", "Wien", "Angriff"]})

Die Nachbarn sind verwandte Wörter, die ein Bag-of-Words-Modell nie miteinander verbinden könnte. Drei Dinge fallen auf:

- **Die Schreibweise schlägt durch.** Viele Nachbarn sind Flexionsformen und Komposita des Suchworts (*Flüchtlings*, *Kriegsflüchtling*, *Gegenangriff*, *Aktienbörse*). Diese Vektoren wurden mit Information über Wortteile trainiert. Das hilft bei der deutschen Morphologie, bedeutet aber auch, dass Wörter mit gemeinsamem Stamm einander ähneln, egal was sie bedeuten.
- **Verwandt ist nicht dasselbe wie ähnlich.** Die Nachbarn von *Wien* sind andere österreichische Städte und Wiener Bezirke, keine Synonyme für Wien; unter den Nachbarn von *Kanzlerin* stehen *Merkel*, aber auch Politikerinnen anderer Parteien. Wörter, die in denselben *Kontexten* vorkommen, liegen nahe beieinander, und Alternativen teilen sich Kontexte. Für die Suche kann das helfen oder schaden.
- **Die Vektoren sind in der Zeit eingefroren.** Sie wurden aus einer allgemeinen Sammlung deutscher Webtexte gelernt, nicht aus unserem Korpus. Was ein Wort in den Nachrichten eines bestimmten Jahres bedeutete - wofür etwa *Obergrenze* oder *Balkanroute* 2015 standen -, wissen sie nur, wenn es in ihren Trainingstexten schon so vorkam.

Und noch einmal das Problem der Komposita:

In [ ]:
# Prüfe, ob es für diese Wörter einen Vektor gibt
for word in ["Flüchtling", "Flüchtlingskrise", "Flüchtlingsunterkunft", "Flüchtlingsobergrenze", "Willkommenskultur", "Sachsen-Anhalt"]:
    print(f"{word:18} hat Vektor: {nlp.vocab[word].has_vector}")

Selbst eine halbe Million Wörter ist ein festes Vokabular. Ein neues Kompositum, das in den Nachrichten eines Jahres geprägt wurde, hat überhaupt keinen Vektor und trägt nichts bei - und im Deutschen werden ständig neue Komposita gebildet.

## Eigene Wortvektoren: word2vec und fastText

Die Vektoren von spaCy wurden auf fremden Texten trainiert. Ein Verfahren, das Wortvektoren lernt, lässt sich aber auch auf dem eigenen Korpus laufen lassen - dann kennen die Vektoren genau die Wörter und Bedeutungen unserer zwei Zeitungen in diesem einen Jahr.

Das bekannteste Verfahren ist **word2vec** (Mikolov et al. 2013). Es ist ein kleines neuronales Netz mit einer Aufgabe, die keine Labels braucht - die Labels stecken im Text selbst (*selbstüberwachtes Lernen*). Es gibt zwei Varianten:

- **CBOW** (*continuous bag of words*) sagt ein Wort aus den Wörtern in einem Fenster um es herum vorher. Die Reihenfolge der Kontextwörter spielt keine Rolle - daher der Name.
- **Skip-gram** macht es umgekehrt: Es sagt aus einem Wort die Wörter in seinem Fenster vorher und gewichtet nahe Wörter stärker als entfernte.

Die Wortvektoren sind ein Nebenprodukt: Es sind die Gewichte, die das Netz dabei lernt. Wörter, die in ähnlichen Kontexten stehen, bekommen ähnliche Gewichte. Laut den Autoren ist CBOW schneller und etwas besser für häufige Wörter, Skip-gram besser für seltene Wörter und kleine Korpora.

**fastText** (Bojanowski et al. 2017) erweitert Skip-gram um **Wortteile**: Jedes Wort wird zusätzlich als Menge seiner Zeichen-N-Gramme dargestellt (*<Fl*, *Flü*, *lüc*, …), und sein Vektor ist die Summe der Vektoren dieser Teile. Das hilft bei Präfixen und Suffixen - und bei Komposita.

Schlagzeilen und Teaser sind für das Training zu wenig Text. Wir nehmen die vollen Artikel beider Zeitungen aus `tm.load_bodies()`, rund 20 Millionen Wörter, zerlegen sie in Sätze und Wörter und trainieren mit dem Paket `gensim`. Das Training dauert auf einem Laptop einige Minuten; die fertigen Modelle werden in `_cache/` gespeichert und bei einem zweiten Lauf von dort geladen.

In [ ]:
# Lade die vollen Artikeltexte und zerlege sie grob in Sätze und Wörter (Wörter mit Bindestrich bleiben ganz)
bodies = tm.load_bodies()
WORD = re.compile(r"[^\W\d_]+(?:-[^\W\d_]+)*")
# sys.intern legt jedes Wort nur einmal im Speicher ab statt bei jedem Vorkommen neu;
# das spart bei 20 Millionen Wörtern rund 1 GB Arbeitsspeicher
sentences = [[sys.intern(w) for w in WORD.findall(s)] for body in bodies for s in re.split(r"(?<=[.!?])\s+", body)]
print(f"{len(sentences):,} Sätze mit {sum(map(len, sentences)):,} Wörtern")


# Trainiere ein Modell oder lade es aus dem Cache
def train(name: str, model_class, **options):
    path = tm.CACHE / f"wordvectors-{name}-{len(sentences)}.model"
    if path.exists():
        return model_class.load(str(path))
    # 150 Dimensionen, Fenster von fünf Wörtern links und rechts, nur Wörter, die mindestens fünfmal vorkommen
    model = model_class(sentences, vector_size=150, window=5, min_count=5, epochs=5, workers=4, seed=0, **options)
    tm.CACHE.mkdir(exist_ok=True)
    model.save(str(path))
    return model


own = {
    "CBOW": train("cbow", Word2Vec, sg=0),
    "Skip-gram": train("skipgram", Word2Vec, sg=1),
    "fastText": train("fasttext", FastText, sg=1),
}
print(f"Vokabular: {len(own['CBOW'].wv):,} Wörter")

Wir vergleichen die Nachbarn einiger Wörter, die 2015/16 eine eigene Bedeutung hatten, mit denen aus dem großen spaCy-Modell:

In [ ]:
# Nächste Nachbarn in den eigenen Modellen und im spaCy-Modell
def own_neighbours(model, word: str, n: int = 6) -> str:
    return ", ".join(w for w, _ in model.wv.most_similar(word, topn=n))


rows = {}
for word in ["Obergrenze", "Balkanroute", "Willkommenskultur", "Merkel"]:
    rows[word] = {"spaCy (fremde Texte)": ", ".join(n.split(" (")[0] for n in neighbours(word, 6)) if nlp.vocab[word].has_vector else "(kein Vektor)"}
    rows[word] |= {name: own_neighbours(model, word) for name, model in own.items()}
pd.DataFrame(rows).T

Die eigenen Vektoren wissen, was im Korpus passiert ist. *Obergrenze* steht in Skip-gram neben *Tageskontingente* und *Flüchtlingszahl*: Es geht um die Obergrenze für Asylanträge, über die Österreich und Deutschland im Winter 2015/16 stritten. *Balkanroute* steht neben *Grenzschließungen* und *Mazedonien*. In den spaCy-Vektoren, gelernt aus allgemeinen Webtexten, steht *Obergrenze* neben *Untergrenze* und *Höchstgrenze* - eine Grenze im Allgemeinen; in unserem Korpus hat das Wort eine sehr spezielle Bedeutung angenommen. Eigene Vektoren messen die **Bedeutung im Korpus**, nicht die im Deutschen allgemein - für die Analyse eines Diskurses ist das oft genau, was wir wollen.

Die drei eigenen Modelle unterscheiden sich so, wie die Theorie es erwarten lässt. CBOW liefert eher Wörter, die an derselben Stelle im Satz stehen könnten (*Obergrenze* - *Maut*, *Mindestsicherung*, *Begrenzung*: alles umstrittene politische Maßnahmen), Skip-gram eher Wörter desselben *Themas*. fastText setzt dagegen stark auf die Schreibweise: Die Nachbarn von *Obergrenze* sind fast alle Komposita mit *-obergrenze*, gleich ob es um Asyl oder Bargeld geht. Das ist der Preis der Wortteile - und gleichzeitig ihr Vorteil, wie das nächste Beispiel zeigt.

Eine klassische Probe für Wortvektoren sind **Analogien**: Wien verhält sich zu Österreich wie Berlin zu was? Gerechnet wird mit Vektoren: *Österreich - Wien + Berlin*.

In [ ]:
# Analogien: a verhält sich zu b wie c zu ?  ->  b - a + c
analogies = [("Wien", "Österreich", "Berlin"), ("Merkel", "Deutschland", "Faymann"), ("Euro", "Europa", "Dollar")]
pd.DataFrame({
    f"{a} : {b} = {c} : ?": {name: ", ".join(w for w, _ in m.wv.most_similar(positive=[b, c], negative=[a], topn=3))
                             for name, m in own.items()}
    for a, b, c in analogies
})

Zwei der drei Analogien gelingen mit word2vec: Die Vektoren haben Beziehungen wie "Hauptstadt von" und "Regierungschef von" als Richtungen im Raum gelernt, ohne dass sie jemand genannt hätte. Die dritte (Euro : Europa = Dollar : *USA*) scheitert in allen Modellen; auf den weiteren Plätzen stehen Wörter derselben Art (andere Kontinente und Länder). fastText greift außerdem zu Wörtern mit ähnlicher Schreibweise (*Österreich-Ungarn*). Analogien sind eine anschauliche Probe, aber keine verlässliche.

Zum Schluss das Problem der Komposita. Ein word2vec-Modell hat für ein Wort, das es nie (oder seltener als `min_count`-mal) gesehen hat, keinen Vektor. fastText setzt einen Vektor aus den Wortteilen zusammen - auch für Wörter, die im Korpus gar nicht vorkommen:

In [ ]:
# Wörter, die selten oder nie im Korpus stehen
for word in ["Flüchtlingsobergrenze", "Asylobergrenzendebatte", "Balkanroutenschließung"]:
    known = word in own["Skip-gram"].wv
    print(f"{word}: im Vokabular von word2vec: {known}")
    print("   fastText-Nachbarn:", own_neighbours(own["fastText"], word, 5))

Keines der drei Wörter hat einen word2vec-Vektor: *Flüchtlingsobergrenze* kommt nur dreimal vor (weniger als `min_count=5`), *Asylobergrenzendebatte* und *Balkanroutenschließung* gar nicht. word2vec kann mit ihnen nichts anfangen. fastText setzt ihre Vektoren aus Zeichen-N-Grammen zusammen (*<As*, *Asy*, *grenz*, *batte*, …), von denen viele auch in bekannten Wörtern vorkommen, und findet sinnvolle Nachbarn. Wortgrenzen innerhalb des Kompositums kennt fastText nicht; es genügt, dass die Teile als Zeichenfolgen bekannt sind. Für eine Sprache, die ständig neue Komposita bildet, ist das ein echter Vorteil.

Weitere Verfahren dieser Familie nennt Han et al.: **GloVe** (Pennington et al. 2014) lernt die Vektoren aus der Matrix der Kookkurrenzen im ganzen Korpus statt aus einzelnen Fenstern, und **Poincaré-Embeddings** betten Wörter in einen hyperbolischen statt einen euklidischen Raum ein, der Hierarchien wie eine Taxonomie besser abbilden kann. Allen gemeinsam ist, dass jedes Wort *einen* Vektor bekommt.

## Dokumentvektoren durch Mittelwertbildung

Der einfachste Dokumentvektor ist der Mittelwert seiner Wortvektoren. spaCy berechnet ihn als `doc.vector`.

In [ ]:
# Drei Umschreibungen derselben Meldung und eine Börsenmeldung
headlines = [
    "Kanzlerin sagt den Ländern mehr Geld für Flüchtlinge zu",
    "Merkel will Bundesländer bei Asylkosten entlasten",
    "Regierungschefin verspricht zusätzliche Mittel zur Unterbringung von Asylbewerbern",
    "Dax schließt mit deutlichem Minus",
]
docs = [nlp(h) for h in headlines]

# Vergleiche die Ähnlichkeit zur ersten Schlagzeile: TF-IDF gegen gemittelte Wortvektoren
tfidf_demo = TfidfVectorizer().fit(headlines + corpus["title"].tolist())
pd.DataFrame(
    {
        "TF-IDF": cosine_similarity(tfidf_demo.transform(headlines))[0].round(2),
        "gemittelte Wortvektoren": [docs[0].similarity(d) for d in docs],
    },
    index=headlines,
).round(2)

Die ersten drei Schlagzeilen sagen fast dasselbe mit anderen Worten. TF-IDF sieht überhaupt keine Überschneidung. Gemittelte Wortvektoren erkennen den Zusammenhang und setzen die Börsenmeldung deutlich darunter - aber die ausführlichste Umschreibung, die dritte, liegt nur knapp über der Mitte. Die Mittelwertbildung verwischt jeden Text in Richtung einer allgemeinen "Nachrichten"-Richtung, ignoriert die Wortreihenfolge und kann nicht gewichten, welche Wörter wichtig sind.

## Sentence Transformer

Ein **Sentence Transformer** ist ein neuronaler Encoder, der gezielt darauf trainiert wurde, Texte mit gleicher Bedeutung auf nahe beieinanderliegende Vektoren abzubilden. Er liest den ganzen Satz im Kontext, sodass der Beitrag eines Worts von seinen Nachbarn abhängt, und er kommt mit unbekannten Komposita zurecht, indem er sie in bekannte Wortteile zerlegt.

In diesem Kurs verwenden wir `paraphrase-multilingual-MiniLM-L12-v2`: klein, schnell und auf Paraphrasen in über fünfzig Sprachen trainiert, darunter Deutsch. Beim ersten Ausführen wird das Modell (etwa 0,5 GB) vom Hugging Face Hub heruntergeladen, danach wird es aus dem lokalen Cache geladen. Eine Grafikkarte (GPU) hilft, ist aber für ein Korpus dieser Größe nicht nötig.

In [ ]:
# Verwende die Grafikkarte, falls vorhanden, sonst die CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)
print("läuft auf", device)

# Kodiere die Beispiel-Schlagzeilen und vergleiche sie mit der ersten
E_demo = encoder.encode(headlines, normalize_embeddings=True)
pd.Series((E_demo @ E_demo[0]).round(2), index=headlines, name="Sentence Transformer").to_frame()

Die drei Paraphrasen liegen jetzt alle klar über der nicht verwandten Börsenmeldung, deren Ähnlichkeit auf null fällt, und die ausführlichste Umschreibung rückt nach vorne. Jede Schlagzeile ist zu einem 384-dimensionalen Einheitsvektor geworden, und die Ähnlichkeit ist wieder ein Skalarprodukt. Das gesamte Korpus zu kodieren dauert auf einer CPU wenige Minuten, mit einer GPU Sekunden:

In [ ]:
# Kodiere alle Schlagzeilen des Korpus
E = encoder.encode(corpus["title"].tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=False)
print(E.shape)

Das Ergebnis ist eine Matrix mit einer Zeile je Schlagzeile und 384 Spalten, den Dimensionen der Embeddings.

## Statisch oder kontextuell?

Alle Wortvektoren bisher - spaCy, word2vec, fastText - sind **statisch**: Ein Wort hat überall denselben Vektor. Das ist ein Problem für Wörter mit mehreren Bedeutungen. Die *Bank*, auf der ein Fußballer sitzt, und die *Bank*, die Zinsen erhöht, bekommen denselben Vektor, einen Mittelwert aus beiden Bedeutungen.

Seit ELMo und **BERT** (Devlin et al. 2019) berechnen **vortrainierte Sprachmodelle** (PLMs) für jedes Wort einen Vektor *im Kontext*: Das Modell liest den ganzen Satz durch viele Schichten eines neuronalen Netzes, und der Vektor eines Worts hängt von seinen Nachbarn ab. Han et al. beschreiben das als **kontextualisierte Wort-Embeddings**, die verschiedene Bedeutungen desselben Worts unterscheiden können. Unser Sentence Transformer ist ein solches Modell; bisher haben wir nur den Mittelwert über alle Wörter eines Satzes verwendet. Die Vektoren der einzelnen Wörter bekommen wir mit `output_value="token_embeddings"`.

Der Encoder zerlegt Wörter in Wortteile (*Bank* bleibt ganz, *Zinsen* wird vielleicht zu *Zins* + *en*). Für den Vektor eines Worts mitteln wir die Vektoren seiner Wortteile; welche Wortteile zu einem Wort gehören, verrät die Zuordnung zu Zeichenpositionen (`offset_mapping`) des Tokenizers.

In [ ]:
# Kontextueller Vektor eines Worts in jedem der Texte (Mittelwert seiner Wortteile, normiert)
def word_in_context(texts: list[str], word: str) -> np.ndarray:
    token_vectors = encoder.encode(texts, output_value="token_embeddings", show_progress_bar=False)
    out = []
    for text, vecs in zip(texts, token_vectors):
        offsets = encoder.tokenizer(text, return_offsets_mapping=True, truncation=True,
                                    max_length=encoder.max_seq_length)["offset_mapping"]
        start, end = re.search(rf"\b{word}\b", text).span()
        parts = [i for i, (a, b) in enumerate(offsets) if b > a and a >= start and b <= end]
        v = vecs[parts].mean(dim=0).cpu().numpy()
        out.append(v / np.linalg.norm(v))
    return np.array(out)


# Vier Sätze, zwei Bedeutungen von "Bank"
bank = ["Die Bank erhöht die Zinsen für Sparer.", "Die Bank meldet einen Verlust im dritten Quartal.",
        "Der Trainer setzt den Stürmer auf die Bank.", "Nach der Pause saß er wieder auf der Bank."]
V = word_in_context(bank, "Bank")
pd.DataFrame(V @ V.T, index=bank, columns=[f"Satz {i + 1}" for i in range(len(bank))]).round(2)

Ein statisches Modell gäbe hier überall 1,00 aus: viermal dasselbe Wort, viermal derselbe Vektor. Die kontextuellen Vektoren von *Bank* sind innerhalb einer Bedeutung ähnlicher als zwischen den Bedeutungen - nicht in jedem Paar mit großem Abstand, aber der Vektor hängt erkennbar vom Kontext ab. Gilt das auch im Korpus? *Bank* kommt in Wirtschaftsmeldungen vor und in Sportmeldungen, wo damit fast immer die Ersatzbank gemeint ist. Wir verwenden die Ressortgruppe als Näherung für die Bedeutung und fragen: Wie viele der fünf nächsten Nachbarn eines *Bank*-Vektors stammen aus derselben Ressortgruppe?

In [ ]:
# Alle Schlagzeilen mit Teaser, die "Bank" enthalten, aus Wirtschaft und Sport
mask = corpus["text"].str.contains(r"\bBank\b") & corpus["section_group"].isin(["Wirtschaft", "Sport"])
bank_texts = corpus.loc[mask, ["text", "section_group"]].reset_index(drop=True)
V = word_in_context(bank_texts["text"].tolist(), "Bank")

# Für jede Nennung: Anteil der fünf nächsten Nachbarn aus derselben Ressortgruppe
S = V @ V.T
np.fill_diagonal(S, -1)
nearest = np.argsort(-S, axis=1)[:, :5]
group = bank_texts["section_group"].to_numpy()
same = (group[nearest] == group[:, None]).mean(axis=1)
pd.DataFrame({
    "Nennungen": bank_texts["section_group"].value_counts(),
    "Nachbarn aus derselben Gruppe": pd.Series(same).groupby(group).mean(),
    "erwartet bei Zufall": bank_texts["section_group"].value_counts(normalize=True),
}).style.format({"Nachbarn aus derselben Gruppe": "{:.0%}", "erwartet bei Zufall": "{:.0%}"})

Die *Bank* aus dem Sport ist im Korpus selten. Wären die Vektoren blind für die Bedeutung, wären die Nachbarn einer Sport-*Bank* nur zu einem kleinen Teil wieder Sport - so selten wie die Sport-*Bank* insgesamt. Tatsächlich stammen fast alle Nachbarn einer Sport-*Bank* wieder aus dem Sport, weit mehr als der Zufall erwarten ließe: Die kontextuellen Vektoren trennen die beiden Bedeutungen fast perfekt, obwohl das Wort in beiden Fällen gleich geschrieben wird. Ein statisches Modell kann das grundsätzlich nicht.

Kontextuelle Embeddings sind die Grundlage fast aller heutigen Sprachmodelle. Han et al. beschreiben, wie sie verwendet werden: **Vortrainieren** auf sehr großen Textmengen, dann **Feintuning** desselben Netzes auf eine konkrete Aufgabe mit wenigen gelabelten Daten. Wir verwenden in diesem Kurs nur die vortrainierten Vektoren, ohne Feintuning - Übung 5 zeigt den nächsten Schritt.

## Semantische Suche

Sind alle Schlagzeilen eingebettet, wird die Suche zu einer Nächste-Nachbarn-Suche: Wir betten die Suchanfrage ein und ordnen die Dokumente nach ihrer Ähnlichkeit. Zum Vergleich führen wir dieselbe Anfrage auch mit TF-IDF durch:

In [ ]:
# Erstelle die TF-IDF-Matrix der Schlagzeilen
tfidf = TfidfVectorizer(min_df=1, token_pattern=r"(?u)\b[^\W\d_][^\W\d_-]*(?:-[^\W\d_]+)*\b")
X = tfidf.fit_transform(corpus["title"])


# Zeige die n besten Treffer der Embedding-Suche und der TF-IDF-Suche nebeneinander
def search(query: str, n: int = 6) -> pd.DataFrame:
    emb = E @ encoder.encode([query], normalize_embeddings=True)[0]
    lex = cosine_similarity(tfidf.transform([query]), X).ravel()
    top_e, top_l = emb.argsort()[::-1][:n], lex.argsort()[::-1][:n]
    return pd.DataFrame({
        "Embedding": [f"{emb[i]:.2f}  {corpus['title'].iloc[i]}" for i in top_e],
        "TF-IDF": [f"{lex[i]:.2f}  {corpus['title'].iloc[i]}" for i in top_l],
    })


search("Ein Autobauer hat bei den Abgaswerten seiner Fahrzeuge betrogen")

Die Anfrage beschreibt den VW-Abgasskandal vom Herbst 2015, ohne die Schlagwörter zu verwenden, mit denen die Zeitungen darüber geschrieben haben (*VW*, *Volkswagen*, *Diesel*, *Abgasaffäre*). TF-IDF findet Treffer nur dort, wo ein Wort der Anfrage wörtlich vorkommt: einige Schlagzeilen mit *Abgaswerten*, aber auch den FC Bayern, der sich *betrogen* fühlt, und Google, das kein *Autobauer* werden will. Die Embedding-Suche findet durchgehend Schlagzeilen über manipulierte Abgaswerte, also das eigentliche Thema, auch wenn sie kein Wort mit der Anfrage teilen. Das ist der zentrale Vorteil von Embeddings: Sie vergleichen **Bedeutung statt Schreibweise**.

Da das Modell mehrsprachig ist, muss die Anfrage nicht einmal auf Deutsch sein:

In [ ]:
# Eine englische Anfrage an die deutschen Schlagzeilen
search("Refugees are walking along the Balkan route towards Germany")

Die englische Anfrage liefert deutsche Schlagzeilen - etwas, das keine wortbasierte Methode kann; TF-IDF findet höchstens Schlagzeilen, die zufällig ein englisches Wort enthalten. Dasselbe funktioniert mit anderen Sprachen, die das Modell kennt:

In [ ]:
# Eine französische Anfrage zum britischen EU-Referendum
search("Le Royaume-Uni vote pour quitter l'Union européenne")

Die französische Anfrage zum britischen EU-Referendum findet die Brexit-Schlagzeilen beider Zeitungen. Ein gemeinsamer Vektorraum für viele Sprachen bedeutet, dass sich Korpora in verschiedenen Sprachen vergleichen und durchsuchen lassen, ohne sie vorher zu übersetzen - etwa Der Standard und die SZ mit einer englischen oder französischen Zeitung desselben Jahres.

## Story Matching, noch einmal

In Skript 04 haben wir Schlagzeilen von Der Standard mit TF-IDF Schlagzeilen der Süddeutschen Zeitung aus demselben Monat zugeordnet und festgestellt, dass die meisten Standard-Schlagzeilen kein nahes Gegenstück hatten. Wir wiederholen genau dasselbe Verfahren mit Embeddings und lesen dieselben Ähnlichkeitsbereiche:

In [ ]:
# Positionen der Standard- und der SZ-Schlagzeilen und der Monat jeder Schlagzeile
std = np.flatnonzero(corpus["outlet"] == "Der Standard")
sz = np.flatnonzero(corpus["outlet"] == "Süddeutsche Zeitung")
month = corpus["month"].to_numpy()

# Für jeden Monat: Ähnlichkeiten der Embeddings und bester SZ-Treffer je Standard-Schlagzeile
rows = []
for m in np.unique(month):
    a, b = std[month[std] == m], sz[month[sz] == m]
    S = E[a] @ E[b].T
    best = S.argmax(axis=1)
    rows.append(pd.DataFrame({"std": a, "sz": b[best], "similarity": S[np.arange(len(a)), best]}))
pairs = pd.concat(rows, ignore_index=True)
pairs["standard"] = corpus["title"].to_numpy()[pairs["std"]]
pairs["sueddeutsche"] = corpus["title"].to_numpy()[pairs["sz"]]

# Zeige je Ähnlichkeitsbereich einige zufällige Paare
for lo, hi in [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)]:
    band = pairs[(pairs["similarity"] >= lo) & (pairs["similarity"] < hi)]
    print(f"\n--- Ähnlichkeit {lo:.1f}-{hi:.1f}: {len(band)} Paare")
    for _, r in band.sample(min(4, len(band)), random_state=0).iterrows():
        print(f"  STD: {r['standard']}\n  SZ:  {r['sueddeutsche']}\n")

Auch hier können wir die Ressorts als zweite Meinung heranziehen, wie in Skript 04: Wie oft stehen die beiden Schlagzeilen eines Paars in derselben Ressortgruppe?

In [ ]:
# Anteil der Paare aus derselben Ressortgruppe je Ähnlichkeitsband (nur Paare mit Gruppe auf beiden Seiten)
group = corpus["section_group"].to_numpy()
both = pairs.assign(group_std=group[pairs["std"]], group_sz=group[pairs["sz"]]).dropna(subset=["group_std", "group_sz"])
bands = pd.cut(both["similarity"], [0, 0.5, 0.6, 0.7, 0.8, 1.01], right=False)
agreement = (both["group_std"] == both["group_sz"]).groupby(bands, observed=True).agg(["mean", "size"])
agreement.columns = ["gleiche Ressortgruppe", "Paare"]
agreement.style.format({"gleiche Ressortgruppe": "{:.0%}"})

Zwei Unterschiede zu Skript 04 fallen auf. Erstens ist die **Skala** eine andere: Die Ähnlichkeiten der Embeddings sind durchwegs höher, weil zwei beliebige Schlagzeilen viel Bedeutung teilen ("jemandem ist irgendwo etwas passiert"); die allermeisten Standard-Schlagzeilen finden im selben Monat einen SZ-Partner über 0,5. Ein Schwellenwert, der für TF-IDF gewählt wurde, lässt sich daher nicht übertragen. Für jede neue Repräsentation muss der Schwellenwert durch Lesen neu kalibriert werden - und hier zeigt das Lesen, dass selbst im obersten Band viele Paare nicht dieselbe Geschichte erzählen. Die Übereinstimmung der Ressortgruppen steigt mit der Ähnlichkeit, ähnlich wie bei TF-IDF.

Zweitens sind die **Fehler** andere. Falsche Zuordnungen bei TF-IDF hatten ein Wort gemeinsam; falsche Zuordnungen bei Embeddings haben eine *Art von Meldung* gemeinsam - zwei verschiedene Unwetter, zwei verschiedene Nachrichten aus Syrien. Die Embeddings finden Paraphrasen, die TF-IDF übersehen hat (*Clooney bei Merkel* und *George Clooney trifft Angela Merkel*), und machen Fehler, die TF-IDF nie machen würde. Keines der beiden Verfahren ist einfach besser; ein robustes Zuordnungsverfahren kombiniert oft beide Signale und ein engeres Zeitfenster, als es die Monatsangaben der SZ erlauben.

## Eine Landkarte des Korpus

**UMAP** projiziert hochdimensionale Vektoren in zwei Dimensionen und versucht dabei, Nachbarn beisammen zu halten. Es ist ein Werkzeug zur Visualisierung: Abstände zwischen weit entfernten Clustern in der Grafik sind nicht aussagekräftig, nur die lokalen Nachbarschaften sind es.

In [ ]:
# Ein fester random_state macht das Layout reproduzierbar, dafür läuft UMAP nur auf
# einem Kern; n_jobs=1 sagt das ausdrücklich, statt eine Warnung auszulösen.
coords = umap.UMAP(n_neighbors=20, min_dist=0.1, metric="cosine", random_state=0, n_jobs=1).fit_transform(E)

# Zeichne die projizierten Schlagzeilen zweimal: eingefärbt nach Zeitung und nach Ressortgruppe
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
for outlet in ["Süddeutsche Zeitung", "Der Standard"]:  # die kleinere Zeitung zuletzt, damit sie sichtbar bleibt
    idx = corpus.index[corpus["outlet"] == outlet]
    ax1.scatter(coords[idx, 0], coords[idx, 1], s=2, alpha=0.3, color=tm.OUTLET_COLOURS[outlet], label=outlet)
group_colours = {"Politik": "#2F6690", "Wirtschaft": "#2E7D5B", "Sport": "#B33A3A", "Panorama": "#946200",
                 "Digital": "#6A4C93"}
rest = corpus["section_group"].isna()
ax2.scatter(coords[rest, 0], coords[rest, 1], s=2, alpha=0.2, color="0.8", label="ohne Gruppe")
for name, colour in group_colours.items():
    idx = corpus.index[corpus["section_group"] == name]
    ax2.scatter(coords[idx, 0], coords[idx, 1], s=2, alpha=0.3, color=colour, label=name)
ax1.set(xticks=[], yticks=[], title="eingefärbt nach Zeitung")
ax2.set(xticks=[], yticks=[], title="eingefärbt nach Ressortgruppe")
for ax in (ax1, ax2):
    ax.legend(markerscale=6, frameon=False, fontsize=8)
fig.suptitle("Embeddings der Schlagzeilen, mit UMAP projiziert", fontweight="bold")
fig.tight_layout()
plt.show()

Vergleiche das mit der LSA-Grafik in Skript 04, in der die Dokumente auf wenige lange Arme verteilt waren. Hier zerfällt das Korpus in viele Nachbarschaften, und die rechte Grafik zeigt, dass sie zu einem guten Teil den Ressorts entsprechen, obwohl der Encoder die Ressorts nie gesehen hat: Sport bildet eine große eigene Region, Digital eine kleinere, Politik verteilt sich auf viele Inseln einzelner Themen und Geschichten. Links sieht man, dass sich die beiden Zeitungen in diesen Nachbarschaften durchmischen: Schlagzeilen über dieselbe Art von Ereignis landen beieinander, egal aus welcher Redaktion sie stammen. Wo die rote Farbe dichter wird, liegen Themen, die Der Standard stärker abdeckt, etwa die Technik seines großen Web-Ressorts.

## Embeddings als Merkmale für einen Klassifikator

Zum Schluss die Aufgabe aus Skript 05. Statt TF-IDF-Merkmalen geben wir das Embedding jeder Schlagzeile in eine logistische Regression, mit derselben zeitlichen Aufteilung in Trainings- und Testdaten. Die logistische Regression kennen wir bereits aus dem Skript `03-data-mining-classification`.

In [ ]:
# Lade Trainings- und Testdaten; ihre Embeddings haben wir oben schon berechnet (Zuordnung über doc_key)
train, test = tm.subject_task()
position = pd.Series(np.arange(len(corpus)), index=corpus["doc_key"])
E_train, E_test = E[position[train["doc_key"]].to_numpy()], E[position[test["doc_key"]].to_numpy()]

# Trainiere die logistische Regression auf den Embeddings
clf = LogisticRegression(max_iter=3000, C=4).fit(E_train, train["label"])
pred_emb = clf.predict(E_test)

# Die beste Pipeline aus Skript 05, hier neu trainiert, damit der Vergleich in sich geschlossen ist.

lesson05 = make_pipeline(
    FeatureUnion([
        ("words", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                                  token_pattern=r"(?u)\b[^\W\d_][^\W\d_-]*(?:-[^\W\d_]+)*\b")),
        ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ]),
    LinearSVC(C=0.25),
).fit(train["title"], train["label"])
pred_tfidf = lesson05.predict(test["title"])

# Vergleiche Accuracy und Makro-F1 der beiden Modelle
pd.DataFrame(
    {
        "Skript 05: TF-IDF + SVM": [accuracy_score(test["label"], pred_tfidf), f1_score(test["label"], pred_tfidf, average="macro")],
        "Embeddings + logistische Regression": [accuracy_score(test["label"], pred_emb), f1_score(test["label"], pred_emb, average="macro")],
    },
    index=["Accuracy", "Makro-F1"],
).T.round(3)

Die beiden Modelle liegen praktisch gleichauf - bemerkenswert, wenn man bedenkt, dass das Embedding-Modell eine einfache logistische Regression ohne Tuning ist, die gegen eine abgestimmte Pipeline antritt, und dass der Encoder nie darauf trainiert wurde, Ressorts zu unterscheiden. Zwei Beobachtungen gelten allgemeiner:

- **Embeddings bringen Vorwissen mit, dünnbesetzte Merkmale lernen auswendig.** Ein Embedding-Modell bringt Wissen aus seinem Vortraining mit und kommt daher mit Wörtern zurecht, die in den gelabelten Daten fehlen. Die Testdaten stammen aus einem späteren Zeitraum mit neuen Meldungen und neuen Namen - genau dort zählt das. TF-IDF-Merkmale kennen nur die Schlagzeilen aus dem Training: Sie erfassen die Namen und Formate, die *damals* die Labels vorhergesagt haben.
- **Sie scheitern auf unterschiedliche Weise** und ergänzen sich daher. Ein Blick auf die Fälle, in denen die beiden Modelle uneins sind, lohnt sich:

In [ ]:
# Wähle die Schlagzeilen aus, bei denen die beiden Modelle unterschiedlich entscheiden
disagree = test.assign(tfidf=pred_tfidf, embedding=pred_emb)
disagree = disagree[disagree["tfidf"] != disagree["embedding"]]
print(f"die beiden Modelle sind sich bei {len(disagree)} von {len(test)} Schlagzeilen der Testdaten uneins")
disagree[["title", "label", "tfidf", "embedding"]].sample(10, random_state=0)

Die Modelle sind sich bei rund einem Drittel der Testdaten uneins, obwohl sie fast gleich oft richtig liegen. Viele Unstimmigkeiten gehen auf die Unschärfe der Ressorts aus Skript 05 zurück - die Deutschkurse des Integrationsministeriums zwischen *Panorama*, *International* und *Inland*, der Asyl-"Notstand" der SPÖ zwischen *Panorama*, *Sonstiges* und *Inland*, eine Sonderkommission des Justizministers nach einem Mord zwischen *Panorama*, *Sonstiges* und *Inland* -, wobei die beiden Modelle auf verschiedenen Seiten landen. Ein Modell, das nur die Schlagzeile sieht, kann weiterhin nicht wissen, welches Ressort einen Artikel betreut hat, und dieser kleine Encoder wurde für Paraphrasen trainiert, nicht für die Klassifikation von Nachrichten. Dass sie verschieden scheitern, ist eine Chance: Übung 1 kombiniert die beiden Signale. Größere Encoder oder das Feintuning eines Encoders auf den gelabelten Schlagzeilen würden weiter helfen - zu echten Kosten an Rechenaufwand und Komplexität.

## Ohne gelabelte Daten: schwache Überwachung

Der Klassifikator oben hat aus knapp 9.000 gelabelten Schlagzeilen gelernt. Han et al. unterscheiden je nach Menge und Art der Labels mehrere Stufen:

- **überwacht**: viele gelabelte Beispiele (Skript 05 und der Abschnitt oben);
- **semi-überwacht**: wenige gelabelte und viele ungelabelte Beispiele;
- **schwach überwacht**: nur sehr wenig Information über die Klassen, im Extremfall **nur ihre Namen**;
- **distant überwacht**: Labels aus einer anderen Quelle, etwa einer Wissensbasis wie Wikipedia.

Embeddings machen die schwache Überwachung einfach: Wenn Schlagzeilen und Klassennamen im selben Raum liegen, ordnen wir jede Schlagzeile der Klasse zu, deren Name ihr am nächsten liegt. Wir probieren zwei Varianten: nur die Namen der Ressorts, und eine kurze Beschreibung jedes Ressorts in einer Zeile. Die Beschreibungen sind deutsche Kurzfassungen der Definitionen, die Skript 07 dem LLM gibt.

In [ ]:
LABELS = tm.SUBJECTS + [tm.OTHER]

# Variante 1: nur die Namen der Klassen
NAMES = {label: label for label in LABELS}
# Variante 2: eine Beschreibung pro Klasse
DESCRIPTIONS = {
    "International": "Außenpolitik: Regierungen, Wahlen, Kriege und Diplomatie im Ausland, EU, Deutschland, USA",
    "Inland": "Österreichische Innenpolitik: Bundesregierung, Parteien, Parlament, Minister, ÖVP, SPÖ, FPÖ",
    "Panorama": "Verbrechen, Unfälle, Katastrophen, Gerichtsprozesse, Polizei, Flüchtlinge an der Grenze",
    "Web": "Technik und Internet: Smartphones, Apps, Spiele, soziale Netzwerke, IT-Sicherheit, Google, Apple",
    "Wirtschaft": "Unternehmen, Märkte, Banken, Börse, Preise, Arbeitsplätze, Steuern, Konjunktur",
    "Sport": "Fußball, Skifahren, Tennis, Formel 1, Spieler, Trainer, Vereine, Tore, Siege",
    "Sonstiges": "Medien, Fernsehen, Wissenschaft, Kultur, Gesundheit, Kommentare, Lebensstil, Reisen",
}


# Zero-Shot: die Klasse, deren Name oder Beschreibung der Schlagzeile am nächsten liegt
def zero_shot(E_docs: np.ndarray, texts: dict) -> np.ndarray:
    E_labels = encoder.encode([texts[label] for label in LABELS], normalize_embeddings=True)
    return E_docs @ E_labels.T  # Ähnlichkeit jeder Schlagzeile zu jeder Klasse


def scores(pred) -> list[float]:
    return [accuracy_score(test["label"], pred), f1_score(test["label"], pred, average="macro", labels=LABELS)]


weak_results = {}
for name, texts in [("nur Klassennamen", NAMES), ("Beschreibungen", DESCRIPTIONS)]:
    weak_results[f"Zero-Shot, {name}"] = scores(np.array(LABELS)[zero_shot(E_test, texts).argmax(axis=1)])
pd.DataFrame(weak_results, index=["Accuracy", "Makro-F1"]).T.round(3)

Ohne ein einziges gelabeltes Beispiel liegt die Zuordnung über die Namen über der Baseline der häufigsten Klasse aus Skript 05 - beim Makro-F1 sogar weit darüber -, aber auch weit unter dem überwachten Modell. Die Beschreibungen helfen deutlich, vor allem beim Makro-F1: Ein Name wie *Panorama* oder *Web* sagt wenig darüber, was das Ressort tatsächlich enthält.

**Selbsttraining** ist ein klassischer Schritt, um aus einem schwachen Klassifikator mehr herauszuholen. Wir labeln die Schlagzeilen der *Trainingsdaten* mit dem Zero-Shot-Verfahren - ohne ihre echten Labels anzusehen - und behalten je Klasse nur die sicherere Hälfte, gemessen am Abstand zwischen der besten und der zweitbesten Klasse. Auf diesen **Pseudo-Labels** trainieren wir eine logistische Regression. Sie kann Muster lernen, die in keiner Beschreibung stehen, aber mit den sicheren Fällen zusammen auftreten.

In [ ]:
# Pseudo-Labels für die Trainingsdaten (die echten Labels bleiben ungenutzt)
S_train = zero_shot(E_train, DESCRIPTIONS)
pseudo = S_train.argmax(axis=1)
top2 = np.sort(S_train, axis=1)[:, -2:]
margin = top2[:, 1] - top2[:, 0]

# Je Klasse die sicherere Hälfte behalten
keep = np.zeros(len(pseudo), dtype=bool)
for k in range(len(LABELS)):
    idx = np.flatnonzero(pseudo == k)
    if len(idx):
        keep[idx[margin[idx] >= np.median(margin[idx])]] = True
print(f"{keep.sum():,} Schlagzeilen mit Pseudo-Label, davon richtig: {(np.array(LABELS)[pseudo[keep]] == train['label'][keep]).mean():.0%}")

# Logistische Regression auf den Pseudo-Labels
self_trained = LogisticRegression(max_iter=3000, C=4).fit(E_train[keep], np.array(LABELS)[pseudo[keep]])
weak_results["Selbsttraining auf Pseudo-Labels"] = scores(self_trained.predict(E_test))

# Zum Vergleich: wenige echte Labels je Klasse (semi-überwacht wäre der nächste Schritt) und alle Labels
rng = np.random.default_rng(0)
for k in [5, 20, 100]:
    idx = np.concatenate([rng.choice(np.flatnonzero(train["label"] == label), k, replace=False) for label in LABELS])
    few = LogisticRegression(max_iter=3000, C=4).fit(E_train[idx], train["label"].iloc[idx])
    weak_results[f"{k} gelabelte Beispiele je Klasse"] = scores(few.predict(E_test))
weak_results["alle Trainingsdaten (Abschnitt oben)"] = scores(pred_emb)
pd.DataFrame(weak_results, index=["Accuracy", "Makro-F1"]).T.round(3)

Die Tabelle ist eine Leiter, und jede Stufe kostet etwas anderes. Eine Zeile Beschreibung je Klasse bringt mehr als die Namen allein. Das Selbsttraining legt noch etwas drauf, obwohl ein guter Teil der Pseudo-Labels falsch ist. Damit liegt die schwache Überwachung zwischen einem Klassifikator mit fünf und einem mit zwanzig echten Beispielen je Klasse. Mit hundert Beispielen je Klasse und erst recht mit allen Trainingsdaten ist sie klar geschlagen: Die Grenzen zwischen den Ressorts von Der Standard lassen sich selbst in einer guten Beschreibung nur grob fassen (in welches Ressort gehört ein Bericht über Flüchtlinge?), und ein Modell lernt sie erst aus Beispielen.

Die praktische Lehre: Schwache Überwachung ist ein guter **Startpunkt**, wenn es noch keine Labels gibt - um ein Korpus vorzusortieren oder um auszuwählen, welche Texte als Erstes von Hand gelabelt werden sollen. Schon zwanzig sorgfältig gelabelte Beispiele je Klasse schlagen hier die Beschreibungen. Skript 07 stellt dieselbe Frage mit einem LLM, das statt einer Zeile eine ausführliche Beschreibung und Beispiele bekommt.

## Zusammenfassung

| Repräsentation | Erfasst | Übersieht |
|---|---|---|
| TF-IDF (Skripten 03-05) | exakte Wörter, Namen, Formate | Synonyme, Paraphrasen, unbekannte Wörter |
| statische Wortvektoren (spaCy) | verwandte Wörter | Kontext, seltene Komposita; eingefrorenes Vokabular |
| eigene Wortvektoren (word2vec) | Bedeutung im eigenen Korpus | Wörter unter `min_count`; brauchen viel Text |
| fastText | unbekannte Komposita über Wortteile | ähnliche Schreibweise wird zu ähnlicher Bedeutung |
| kontextuelle Wortvektoren | verschiedene Bedeutungen desselben Worts | Rechenaufwand; schwer zu interpretieren |
| gemittelte Wortvektoren | grobe Bedeutung des Dokuments | Wortreihenfolge, welche Wörter wichtig sind |
| Sentence Transformer | Bedeutung im Kontext, über Sprachen hinweg | exakte Namen; Schwellenwert muss neu kalibriert werden |

Mit Embeddings lässt sich auch ganz ohne gelabelte Daten klassifizieren (**schwache Überwachung** über die Namen oder Beschreibungen der Klassen). Das ist ein guter Startpunkt, aber kein Ersatz für Beispiele.

## Übungen

1. **Kombiniere die Signale.** Hänge das Embedding an die TF-IDF-Merkmale aus Skript 05 an (`scipy.sparse.hstack`) und trainiere ein gemeinsames Modell. Schlägt es beide?
2. **Sprachübergreifende Suche.** Formuliere fünf Suchanfragen auf Deutsch und übersetze sie ins Englische und in eine weitere Sprache. Wie viele der sechs besten Treffer stimmen zwischen den Sprachen überein? Bei welchen Anfragen hängt das Ergebnis am stärksten von der Sprache ab?
3. **Kalibrierung des Schwellenwerts.** Bewerte 40 Zuordnungen von Schlagzeilen auf Basis der Embeddings als richtig oder falsch und zeichne die Precision in Abhängigkeit vom Schwellenwert der Ähnlichkeit. Kombiniere dann TF-IDF- und Embedding-Ähnlichkeit (etwa ihren Mittelwert): Steigt die Precision?
4. **Ein größerer Encoder.** Probiere `intfloat/multilingual-e5-base` aus (stelle jedem Text `"query: "` voran, wie es die Modellbeschreibung verlangt). Ist die Suche besser, und was kostet das an Zeit?
5. **Feintuning.** Trainiere den Sentence Transformer mit `sentence_transformers` einige Epochen lang auf den gelabelten Schlagzeilen der Trainingsdaten (etwa mit `BatchAllTripletLoss`, sodass Schlagzeilen desselben Ressorts zusammenrücken), und wiederhole den Klassifikator. Wie viel bringt das Feintuning gegenüber den vortrainierten Embeddings?
6. **Fenstergröße.** Trainiere word2vec mit `window=2` und `window=15`. Wie ändern sich die Nachbarn von *Obergrenze* und *Merkel*? Welches Fenster liefert eher Wörter derselben Art, welches eher Wörter desselben Themas?

## Quellen

* Han, J., Pei, J., Tong, H. (2023): *Data Mining: Concepts and Techniques*, 4. Aufl., Kap. 12.1. Morgan Kaufmann.
* Mikolov, T., Chen, K., Corrado, G., Dean, J. (2013): Efficient Estimation of Word Representations in Vector Space. arXiv:1301.3781.
* Bojanowski, P., Grave, E., Joulin, A., Mikolov, T. (2017): Enriching Word Vectors with Subword Information. *TACL* 5, 135-146.
* Pennington, J., Socher, R., Manning, C. D. (2014): GloVe: Global Vectors for Word Representation. *Proceedings of EMNLP 2014*, 1532-1543.
* Devlin, J., Chang, M.-W., Lee, K., Toutanova, K. (2019): BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. *Proceedings of NAACL 2019*, 4171-4186.
* https://radimrehurek.com/gensim/

* https://spacy.io/models/de
* https://sbert.net/
* https://umap-learn.readthedocs.io/
* https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
* https://scikit-learn.org/stable/modules/svm.html
* https://scikit-learn.org/stable/modules/model_evaluation.html